# Understanding `stop_reason` with Claude on AWS Bedrock

Every Claude API response includes a `stop_reason` field that tells you **why the model stopped generating**.

| `stop_reason` | Meaning |
|---|---|
| `end_turn` | Model finished naturally |
| `max_tokens` | Hit the `max_tokens` limit |
| `tool_use` | Model wants to call a tool |
| `stop_sequence` | Hit a custom stop sequence |

## Setup
Install the Anthropic library with Bedrock support:

In [2]:
!pip install anthropic

In [3]:
import anthropic

# AnthropicBedrock uses your AWS credentials (env vars or ~/.aws/credentials)
# Required: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_REGION (or boto3 profile)
client = anthropic.AnthropicBedrock(
    aws_region="ap-southeast-1",  # change to your region
)

MODEL = "apac.anthropic.claude-3-5-sonnet-20241022-v2:0"
print("Client ready.")

Client ready.


---
## Case 1: `end_turn` — Model finishes naturally

In [4]:
response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer briefly."}],
)

print(f"stop_reason : {response.stop_reason}")
print(f"response    : {response.content[0].text}")

ModuleNotFoundError: No module named 'botocore'

---
## Case 2: `max_tokens` — Response was cut off

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=10,   # intentionally tiny
    messages=[{"role": "user", "content": "Write a 500-word essay about the ocean."}],
)

print(f"stop_reason : {response.stop_reason}")
print(f"response    : {response.content[0].text}")

if response.stop_reason == "max_tokens":
    print("\n⚠️  Response was truncated. Increase max_tokens to get the full answer.")

---
## Case 3: `tool_use` — Model wants to call a tool

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"],
        },
    }
]

response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    tools=tools,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
)

print(f"stop_reason : {response.stop_reason}")

if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")
    print(f"tool called : {tool_block.name}")
    print(f"tool input  : {tool_block.input}")

---
## Case 4: `stop_sequence` — Hit a custom stop marker

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    stop_sequences=["STOP"],
    messages=[{
        "role": "user",
        "content": "Count from 1 to 10, then write STOP, then keep going."
    }],
)

print(f"stop_reason : {response.stop_reason}")
print(f"response    : {response.content[0].text}")

---
## Putting It Together: Agentic Loop

`stop_reason` is the **loop control** in any agentic workflow.
Keep running while the model asks to use tools; stop when it says `end_turn`.

In [ ]:
import time
from anthropic import RateLimitError


def call_with_retry(client, max_retries=5, **kwargs):
    """Wraps client.messages.create with exponential backoff on RateLimitError."""
    for attempt in range(max_retries):
        try:
            return client.messages.create(**kwargs)
        except RateLimitError:
            wait = 2 ** attempt  # 1s → 2s → 4s → 8s → 16s
            print(f"  Rate limited. Waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)
    raise RuntimeError("Exceeded max retries due to rate limiting.")


def fake_calculator(expression: str) -> str:
    try:
        return str(eval(expression))  # noqa: S307 — demo only
    except Exception as e:
        return f"Error: {e}"


calc_tool = [
    {
        "name": "calculator",
        "description": "Evaluate a math expression.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. '3 * (7 + 2)'"}
            },
            "required": ["expression"],
        },
    }
]

messages = [{"role": "user", "content": "What is (123 * 456) + 789?"}]

print("--- Agentic Loop Start ---")
while True:
    response = call_with_retry(
        client,
        model=MODEL,
        max_tokens=512,
        tools=calc_tool,
        messages=messages,
    )

    print(f"stop_reason: {response.stop_reason}")

    if response.stop_reason == "end_turn":
        print(f"Final answer: {response.content[0].text}")
        break

    if response.stop_reason == "tool_use":
        tool_block = next(b for b in response.content if b.type == "tool_use")
        result = fake_calculator(tool_block.input["expression"])
        print(f"  → called {tool_block.name}({tool_block.input}) = {result}")

        messages.append({"role": "assistant", "content": response.content})
        messages.append({
            "role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": result,
            }],
        })

print("--- Agentic Loop End ---")

---
## Bonus: Parallel Tool Use — Multiple Tools in One `tool_use` Response

When Claude needs independent information simultaneously, it returns **multiple `tool_use` blocks**
in a single response (still one `stop_reason: tool_use`).

You must execute **all** of them and return **all** results in the next user message.

In [ ]:
import time
from anthropic import RateLimitError

# --- Fake tool implementations ---
def get_weather(city: str) -> str:
    return f"Sunny, 28°C in {city}"

def get_population(city: str) -> str:
    data = {"Tokyo": "13.96 million", "Paris": "2.16 million", "Sydney": "5.31 million"}
    return data.get(city, "Unknown")

def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    return f"1 {from_currency} = 0.0067 {to_currency} (fake rate)"

TOOL_REGISTRY = {
    "get_weather": get_weather,
    "get_population": get_population,
    "get_exchange_rate": get_exchange_rate,
}

# --- Tool definitions ---
multi_tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
    {
        "name": "get_population",
        "description": "Get the population of a city.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
    {
        "name": "get_exchange_rate",
        "description": "Get the exchange rate between two currencies.",
        "input_schema": {
            "type": "object",
            "properties": {
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"},
            },
            "required": ["from_currency", "to_currency"],
        },
    },
]

# --- Prompt that forces parallel tool calls ---
messages = [{
    "role": "user",
    "content": (
        "I'm planning a trip to Tokyo. "
        "What's the weather there, what's the population, "
        "and what's the JPY to USD exchange rate?"
    ),
}]

print("--- Parallel Tool Use Demo ---\n")

def call_with_retry(client, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return client.messages.create(**kwargs)
        except RateLimitError:
            wait = 2 ** attempt
            print(f"  Rate limited. Waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)
    raise RuntimeError("Exceeded max retries.")

while True:
    response = call_with_retry(
        client,
        model=MODEL,
        max_tokens=1024,
        tools=multi_tools,
        messages=messages,
    )

    print(f"stop_reason : {response.stop_reason}")
    print(f"blocks in response: {[b.type for b in response.content]}\n")

    if response.stop_reason == "end_turn":
        final_text = next(b.text for b in response.content if b.type == "text")
        print(f"Final answer:\n{final_text}")
        break

    if response.stop_reason == "tool_use":
        tool_blocks = [b for b in response.content if b.type == "tool_use"]
        print(f"Claude wants to call {len(tool_blocks)} tool(s) in parallel:")

        # Execute ALL tool calls and collect ALL results
        tool_results = []
        for tb in tool_blocks:
            fn = TOOL_REGISTRY[tb.name]
            result = fn(**tb.input)
            print(f"  ✓ {tb.name}({tb.input}) → {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tb.id,
                "content": result,
            })

        # Send ALL results back in a single user message
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        print()

print("\n--- Done ---")

---
## Summary

```
stop_reason == "end_turn"      → done, read response.content
stop_reason == "max_tokens"    → truncated, raise max_tokens or paginate
stop_reason == "tool_use"      → execute the tool(s), send result(s) back, loop
stop_reason == "stop_sequence" → hit your custom marker, process partial output
```

### Parallel Tool Use — Key Rules

```
1. One stop_reason == "tool_use" can contain MULTIPLE tool_use blocks.
2. Execute ALL of them.
3. Return ALL results in a single user message (list of tool_result dicts).
4. Missing even one result causes an error on the next API call.
```